# Chapter 02: Exploratory Data Analysis

## Engineering Question
> How do the statistical properties, high dimensionality, and class overlap in the NSL-KDD dataset impact the design and performance of unsupervised anomaly detection models?

---

### Objective
The objective of this notebook is to perform a comprehensive exploratory data analysis (EDA) of the NSL-KDD dataset. We will analyze the distribution of numerical and categorical features, evaluate feature correlations, inspect class imbalance, map individual attack types to higher-level threat families, and preview a 2D PCA projection of the feature space. This analysis will guide our engineering decisions during the modeling phase.

## Background / Theory

### High-Dimensional Feature Space
The NSL-KDD dataset has 41 active features, creating a high-dimensional space. In high-dimensional spaces, data points become extremely sparse, and the Euclidean distance between any two points converges (often referred to as the 'Curse of Dimensionality'). This makes distance-based clustering algorithms (like DBSCAN) struggle to distinguish noise points from dense clusters without fine-tuned hyperparameter limits.

### Mixed Attributes, Scaling, and Encoding
- **Categorical Features**: Machine learning models require numerical matrices. Features like `protocol_type`, `service`, and `flag` represent protocol states and must be encoded.
- **Feature Scaling**: Numerical attributes range from single-digit rates (e.g. `serror_rate` between 0 and 1) to large packet sizes (e.g. `src_bytes` up to billions). Without scaling, attributes with large numerical scales dominate distance calculations, causing models like DBSCAN and Autoencoders to fail. We will use standardization ($z$-score scaling) to give all features zero mean and unit variance.

### PCA for Visualization vs. Training
Principal Component Analysis (PCA) is an unsupervised linear dimensionality reduction technique. It projects features onto orthogonal axes that maximize explained variance. While extremely useful for projecting 41 features down to a 2D or 3D coordinate space for human visualization, training anomaly models directly on low-dimensional PCA coordinates represents a major risk. Anomaly detectors like Isolation Forest and Autoencoders rely on high-dimensional feature details to identify subtle anomalies; throwing away high-frequency principal components can lead to high false-negative rates.

## Imports

All imports originate from standard libraries, Plotly, or our modularized project backend (`src` / `configs`).

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

# Ensure project root is in path for imports
sys.path.append(os.path.abspath("...." if ".." in sys.path else ".."))

from configs import config
from src.data.dataset import load_train_data, dataset_summary
from src.data.preprocessing import prepare_training_data, get_binary_labels
from src.visualization.pca import compute_pca, prepare_pca_dataframe
from src.visualization.plotly_plots import pca_2d_plot

# Set Plotly default template
pio.templates.default = config.PLOT_TEMPLATE

## Dataset Statistics

First, we load the training set and display its dimensions, missing values count, and duplicates count.

In [2]:
train_df = load_train_data()
summary = dataset_summary(train_df)

print(f"Dataset dimensions: {train_df.shape}")
print(f"Missing values count: {summary['missing_values']}")
print(f"Duplicate rows count: {summary['duplicate_rows']}")

Dataset dimensions: (125973, 42)
Missing values count: 0
Duplicate rows count: 0


## Feature Categories

Partition the feature space into numerical and categorical variables.

In [3]:
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
if 'label' in categorical_cols:
    categorical_cols.remove('label')

numerical_cols = train_df.select_dtypes(exclude=['object']).columns.tolist()

print(f"Categorical Features: {categorical_cols}")
print(f"Numerical Features Count: {len(numerical_cols)}")

Categorical Features: ['protocol_type', 'service', 'flag']
Numerical Features Count: 38


## Categorical Features Analysis

Inspect the category distributions for `protocol_type` and the most frequent `service` and `flag` attributes.

In [4]:
proto_counts = train_df['protocol_type'].value_counts().reset_index()
proto_counts.columns = ['Protocol', 'Count']

fig_proto = px.bar(
    proto_counts,
    x='Protocol',
    y='Count',
    title='Protocol Type Distribution',
    color='Protocol'
)
fig_proto.update_layout(showlegend=False, width=650, height=400)
fig_proto.show()

Let's look at the top 10 most common network services in the dataset.

In [5]:
print("=== Top 10 Services ===")
print(train_df['service'].value_counts().head(10))

=== Top 10 Services ===
service
http        40338
private     21853
domain_u     9043
smtp         7313
ftp_data     6860
eco_i        4586
other        4359
ecr_i        3077
telnet       2353
finger       1767
Name: count, dtype: int64


## Numerical Features Analysis

Describe the numerical attributes. We look at the packet byte counts `src_bytes` and `dst_bytes` to evaluate skewness.

In [6]:
print("=== Descriptive Statistics for Bytes ===")
print(train_df[['src_bytes', 'dst_bytes']].describe())

=== Descriptive Statistics for Bytes ===
          src_bytes     dst_bytes
count  1.259730e+05  1.259730e+05
mean   4.556674e+04  1.977911e+04
std    5.870331e+06  4.021269e+06
min    0.000000e+00  0.000000e+00
25%    0.000000e+00  0.000000e+00
50%    4.400000e+01  0.000000e+00
75%    2.760000e+02  5.160000e+02
max    1.379964e+09  1.309937e+09


## Correlation Analysis

Evaluate correlations within our feature matrix. Features with correlation coefficients $>0.90$ are redundant and can be documented.

In [7]:
corr = train_df[numerical_cols].corr()
high_corr = []
for i in range(len(numerical_cols)):
    for j in range(i+1, len(numerical_cols)):
        if abs(corr.iloc[i, j]) > 0.9:
            high_corr.append((numerical_cols[i], numerical_cols[j], corr.iloc[i, j]))

print("Highly correlated feature pairs (> 0.9):")
for f1, f2, c in high_corr:
    print(f"{f1} <-> {f2}: {c:.4f}")

Highly correlated feature pairs (> 0.9):
num_compromised <-> num_root: 0.9988
serror_rate <-> srv_serror_rate: 0.9933
serror_rate <-> dst_host_serror_rate: 0.9794
serror_rate <-> dst_host_srv_serror_rate: 0.9811
srv_serror_rate <-> dst_host_serror_rate: 0.9776
srv_serror_rate <-> dst_host_srv_serror_rate: 0.9863
rerror_rate <-> srv_rerror_rate: 0.9890
rerror_rate <-> dst_host_rerror_rate: 0.9267
rerror_rate <-> dst_host_srv_rerror_rate: 0.9644
srv_rerror_rate <-> dst_host_rerror_rate: 0.9178
srv_rerror_rate <-> dst_host_srv_rerror_rate: 0.9702
dst_host_serror_rate <-> dst_host_srv_serror_rate: 0.9851
dst_host_rerror_rate <-> dst_host_srv_rerror_rate: 0.9247


Let's visualize the correlation matrix for a subset of numerical columns using an interactive Plotly Heatmap.

In [8]:
subset_cols = numerical_cols[:15]
fig_corr = px.imshow(
    train_df[subset_cols].corr(),
    labels=dict(color="Correlation"),
    x=subset_cols,
    y=subset_cols,
    title="Correlation Heatmap (Subset of Numerical Features)",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1
)
fig_corr.update_layout(width=700, height=600)
fig_corr.show()

## Class Imbalance Analysis

Convert labels to a binary flag (`normal -> 0`, `anomaly -> 1`) and evaluate the class distribution.

In [9]:
y_train_binary = get_binary_labels(train_df['label'])
binary_counts = y_train_binary.value_counts().reset_index()
binary_counts.columns = ['Is_Anomaly', 'Count']
binary_counts['Class'] = binary_counts['Is_Anomaly'].map({0: 'Normal (0)', 1: 'Anomaly (1)'})

fig_imbalance = px.bar(
    binary_counts,
    x='Class',
    y='Count',
    title='Class Imbalance: Normal vs Anomaly',
    color='Class',
    labels={'Class': 'Traffic Class', 'Count': 'Records'}
)
fig_imbalance.update_layout(showlegend=False, width=600, height=400)
fig_imbalance.show()

## Attack Categories & Traffic Families

Group individual attack types into higher-level threat categories: DoS, Probe, U2R, R2L, or Normal.

In [10]:
attack_mapping = {
    'normal': 'Normal',
    
      # DoS
    'neptune': 'DoS', 'back': 'DoS', 'land': 'DoS', 'pod': 'DoS', 'smurf': 'DoS', 'teardrop': 'DoS',
    'mailbomb': 'DoS', 'processtable': 'DoS', 'udpstorm': 'DoS', 'apache2': 'DoS', 'worm': 'DoS',
    
      # Probe
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe', 'mscan': 'Probe', 'saint': 'Probe',
    
      # U2R
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'multihop': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R',
    'sqlattack': 'U2R', 'xterm': 'U2R', 'ps': 'U2R',
    
      # R2L
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'phf': 'R2L', 'spy': 'R2L', 'warezclient': 'R2L',
    'warezmaster': 'R2L', 'sendmail': 'R2L', 'named': 'R2L', 'snmpgetattack': 'R2L', 'snmpguess': 'R2L',
    'xlock': 'R2L', 'xsnoop': 'R2L', 'httptunnel': 'R2L'
}

train_df['attack_family'] = train_df['label'].map(attack_mapping).fillna('Other Attack')
family_counts = train_df['attack_family'].value_counts().reset_index()
family_counts.columns = ['Family', 'Count']

fig_families = px.bar(
    family_counts,
    x='Family',
    y='Count',
    title='Distribution of Network Traffic Families',
    color='Family',
    labels={'Family': 'Traffic Family', 'Count': 'Records'}
)
fig_families.update_layout(showlegend=False, width=700, height=450)
fig_families.show()

## PCA Preview

Apply our standardized preprocessing pipeline to standard scale features, execute 2D PCA project, and visualize the output using the modularized Plotly PCA scatter plot.

In [11]:
# Run preprocessing
train_df_clean = train_df.drop(columns=['attack_family'], errors='ignore')
x_train, y_train = prepare_training_data(train_df_clean)
y_train_binary = get_binary_labels(y_train)

# Compute 2D PCA
pca_coords, pca_obj = compute_pca(x_train, n_components=2)
pca_plot_df = prepare_pca_dataframe(pca_coords, y_train_binary)
pca_plot_df['Class'] = pca_plot_df['Label'].map({0: 'Normal', 1: 'Anomaly'})

# Render standard 2D PCA plot
fig_pca = pca_2d_plot(pca_plot_df, color='Class', title='2D PCA Projection of Network Traffic')
fig_pca.show()

## Key Findings

- **Feature Correlation**: We discovered extremely strong positive correlation pairs (e.g. `num_compromised` vs `num_root` at 0.999; and `serror_rate` vs `dst_host_serror_rate` at 0.979). This indicates significant redundancy, which tree-based estimators (like Isolation Forest) handle easily, but which can increase training times for neural network encoders.
- **Traffic Mix**: TCP dominates the traffic, representing over 81% of records. HTTP is the most frequent service, making up 32% of connections.
- **Class Overlap**: As shown in the 2D PCA projection, Normal and Anomaly classes are heavily mixed, and do not form distinct, widely separated clusters. This highlights that basic Euclidean distance clustering (like DBSCAN) will face challenges separating noise from density.

## Engineering Notes

### Why DBSCAN may struggle before modeling?
DBSCAN computes spatial density based on pairwise Euclidean distances. In the high-dimensional feature space, distances become uniform. Furthermore, network traffic contains heavily overlapping boundaries where attack packet headers mimic normal TCP states. DBSCAN will likely struggle, either grouping anomalous traffic into the normal cluster or treating everything as sparse outliers unless the threshold variables are set carefully.

### Why do attack classes overlap?
Many network attacks (such as Port Scanning or slow DoS) use standard protocols and service channels (HTTP, SMTP). The packets look syntactically normal, only differing in frequency or timing. This results in overlapping regions in coordinate space.

### How do network datasets differ from business datasets?
Business datasets (such as e-commerce transactions) typically present low anomaly rates ($<1\%$) and stable schemas. Network intrusion datasets present extremely high attack proportions (46% in our case) and exhibit severe feature drift due to continuously evolving malicious protocols.

## Interview Questions

1. **Why shouldn't you train an Isolation Forest model directly on low-dimensional PCA coordinates?**
   * *Guideline*: Explain that PCA projects data along axes of maximum variance, which typically represent major patterns of normal traffic. Anomaly detection models rely on sparse, low-variance directions or detailed high-dimensional feature subsets to detect outliers, which PCA might discard as 'noise'.

2. **Why is StandardScaler (standardization) preferred over MinMaxScaler (normalization) for distance-based anomaly models?**
   * *Guideline*: Network traffic features (like packet size or duration) have extreme values and long tails. Normalization bounds all features to `[0, 1]`, which crushes the normal variance into a narrow band. Standardization is robust to outliers, preserving variance scales.

3. **How does high feature correlation affect Autoencoders compared to tree-based Isolation Forests?**
   * *Guideline*: High correlation represents redundancy. Isolation Forest handles redundancy easily because it splits random features. For Autoencoders, highly correlated inputs can cause the network to learn a trivial mapping (e.g. duplicating weight states) rather than identifying meaningful low-dimensional structures.

4. **How does the high proportion of anomalies (46%) affect the contamination parameter in Isolation Forest?**
   * *Guideline*: Contamination represents the expected fraction of outliers in the dataset. If we set it too low, the decision path threshold will be too conservative, missing anomalies. Setting it close to 0.50 aligns with the actual dataset statistics.

5. **Why does DBSCAN suffer from the Curse of Dimensionality, and how does this affect epsilon tuning?**
   * *Guideline*: In high dimensions, the volume of space increases exponentially, making data points look equally distant from one another. This reduces distance variance, making it very difficult to choose a single epsilon radius that separates clusters.

## Key Takeaways
- Categorical values represent a significant portion of traffic states and must be encoded.
- Normal and anomalous records show significant spatial overlap in 2D space.
- Redundancy exists in numerical features (highly correlated protocol error rates).

## Future Improvements
- **Feature Selection**: Implement a feature selection step (such as removing one of the features in highly correlated pairs) to reduce dimensionality and training overhead for neural networks.

## Conclusion

We have explored feature distributions, evaluated correlation metrics, mapped attack families, and previewed the PCA projection. We are now ready to clean, encode, scale, and save our datasets.

## Next Notebook

Proceed to the next chapter: [Data Preprocessing](file:///c:/Projects/Network%20anomoly%20detection/notebooks/03_data_preprocessing.ipynb)